# Trial Inspector

Set one `generation_run_id` + `trial_id` below and re-run every cell (Run All) to see
everything about that trial in one place: the canonical scene/character data, the U/S/P
dialogue transcripts, the H1 judge prompts (with a check for whether they're really
condition-blind), and the full H1 metric breakdown per condition.

This is a drill-down tool for one trial at a time, separate from `results_dashboard.ipynb`
(which aggregates across trials/conditions).

In [1]:
import glob
import json
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # this notebook lives in notebooks/, one level under the repo root
H1_PATH = REPO_ROOT / "output" / "evaluation" / "h1_consistency.jsonl"
TRAJECTORY_GLOB = str(REPO_ROOT / "output" / "record" / "*" / "trajectory" / "*.jsonl")
ALL_ACTIONS_GLOB = str(REPO_ROOT / "output" / "record" / "*" / "all" / "*.jsonl")

CONDITION_ORDER = ["U", "S", "P"]


def load_jsonl(path):
    with open(path, encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

In [2]:
def find_trajectory_records(generation_run_id, trial_id):
    """Search every trajectory file under output/record/*/trajectory/, not one hardcoded
    path, so this keeps working if more scenes/titles/models get added later."""
    matches = {}
    for path in sorted(glob.glob(TRAJECTORY_GLOB)):
        for record in load_jsonl(path):
            if record.get("generation_run_id") == generation_run_id and record.get("trial_id") == trial_id:
                matches[record["context_condition"]] = record
    return matches


def find_generation_calls(generation_run_id, trial_id, condition):
    """One line in the `all` record file per (trial, condition), saved once the whole
    trial finishes; its `record` list holds one {prompt, response, duration} dict per
    round, in round order (record[0] = round 1). Search every `all` file, not one
    hardcoded path, for the same reason as find_trajectory_records()."""
    for path in sorted(glob.glob(ALL_ACTIONS_GLOB)):
        for line in load_jsonl(path):
            if (
                line.get("generation_run_id") == generation_run_id
                and line.get("trial_id") == trial_id
                and line.get("context_condition") == condition
            ):
                return line["record"]
    return None


def find_h1_rows(generation_run_id, trial_id):
    by_condition = {condition: [] for condition in CONDITION_ORDER}
    for row in load_jsonl(H1_PATH):
        if row.get("generation_run_id") == generation_run_id and row.get("trial_id") == trial_id:
            by_condition[row["context_condition"]].append(row)
    for condition in by_condition:
        by_condition[condition].sort(key=lambda r: r["round"])
    return by_condition

## Set the trial to inspect

In [3]:
GENERATION_RUN_ID = "cb_hp_019_luna_neville-1788923920758488000"
TRIAL_ID = "cb_hp_019_luna_neville_t10"

trajectory_records = find_trajectory_records(GENERATION_RUN_ID, TRIAL_ID)
h1_rows = find_h1_rows(GENERATION_RUN_ID, TRIAL_ID)
generation_calls = {c: find_generation_calls(GENERATION_RUN_ID, TRIAL_ID, c) for c in CONDITION_ORDER}

missing_trajectory = [c for c in CONDITION_ORDER if c not in trajectory_records]
missing_h1 = [c for c in CONDITION_ORDER if not h1_rows[c]]
missing_generation = [c for c in CONDITION_ORDER if not generation_calls[c]]
if missing_trajectory:
    raise ValueError(f"No trajectory record found for conditions: {missing_trajectory}")
if missing_h1:
    raise ValueError(f"No H1 rows found for conditions: {missing_h1}")
if missing_generation:
    raise ValueError(f"No generation-call trace ('all' record) found for conditions: {missing_generation}")

print(f"Found trial {TRIAL_ID!r} in run {GENERATION_RUN_ID!r}: all of U/S/P present.")
print({c: len(h1_rows[c]) for c in CONDITION_ORDER}, "rounds scored per condition")

Found trial 'cb_hp_019_luna_neville_t10' in run 'cb_hp_019_luna_neville-1788923920758488000': all of U/S/P present.
{'U': 10, 'S': 10, 'P': 10} rounds scored per condition


### 1. Scene information

In [4]:
reference = trajectory_records["U"]
scene = reference["scene"]
player = reference["player"]
target = reference["target"]

print("[Scene]")
for key in ("event", "time", "location", "description"):
    print(f"{key.capitalize()}: {scene.get(key)}")

print()
print(f"[Player — {player['name']}]")
for key in ("position", "states", "description"):
    print(f"{key.capitalize()}: {player.get(key)}")

print()
print(f"[Target — {target['name']}]")
for key in ("position", "states", "description", "self_belief", "environment_belief"):
    print(f"{key.capitalize()}: {target.get(key)}")

print()
print("[Action Reference]")
print(reference["action_reference"])

# docs/research-contract.md §5: fields-and-values(U) = fields-and-values(S) = fields-and-values(P).
# Cheap, concrete check that this trial actually respects it, not just an assumption.
equivalent = all(
    (trajectory_records[c]["scene"], trajectory_records[c]["target"], trajectory_records[c]["action_reference"])
    == (scene, target, reference["action_reference"])
    for c in CONDITION_ORDER
)
print()
print(f"Scene/character/action_reference identical across U/S/P: {equivalent}")

[Scene]
Event: The Secret Meeting
Time: Late evening
Location: The Shrieking Shack, Hogwarts
Description: The atmosphere is eerie as the wind howls around the Shrieking Shack. The moon is hidden behind clouds, casting a mysterious glow over the scene. A secret meeting is about to take place between two key characters.

[Player — Luna Lovegood]
Position: Standing near the entrance of the Shrieking Shack, looking around warily.
States: Luna is feeling a mix of anticipation and caution, eager to discuss the latest developments but aware of the risks involved in meeting in such a secluded place.
Description: Luna is a quirky and insightful student, known for her unique perspective on the magical world. She is wearing her characteristic radish earrings and is excited about the upcoming meeting.

[Target — Neville Longbottom]
Position: Hiding behind a large rock, keeping an eye out for any sign of trouble.
States: Neville is nervous but determined, ready to support Luna and the plan they hav

In [5]:
def format_interaction(record):
    player_name = record["player"]["name"]
    target_name = record["target"]["name"]
    lines = []
    for turn in record["turns"]:
        lines.append(f"Round {turn['round']}")
        lines.append(f"{player_name}: {turn['player_dialogue']}")
        lines.append(f"{target_name}: {turn['npc_dialogue']}")
    return "\n".join(lines)


def print_generation_prompt_structure(condition):
    """Round 1's actual filled generation prompt for this condition -- this is where
    U/S/P genuinely differ in structure (unlike the H1 judge prompt, which is
    deliberately condition-blind). Later rounds follow the same structure with the
    rolling Observation/memory content updated each round; not reprinted per round
    here since the structure itself doesn't change, only that one field's content."""
    first_call = generation_calls[condition][0]
    print(first_call["prompt"])

### 2. U — interaction

**Prompt structure used to generate U's responses** (round 1, representative):

In [6]:
print_generation_prompt_structure("U")

Please act as Neville Longbottom and respond using the tone, manner and vocabulary Neville Longbottom would use. Use the following context to produce one appropriate line of dialogue.
The Secret Meeting
Late evening
The Shrieking Shack, Hogwarts
The atmosphere is eerie as the wind howls around the Shrieking Shack. The moon is hidden behind clouds, casting a mysterious glow over the scene. A secret meeting is about to take place between two key characters.
Neville Longbottom
Neville is a brave and loyal friend to Harry Potter and his classmates. He has been practicing his magical skills and is ready to contribute to their cause.
Hiding behind a large rock, keeping an eye out for any sign of trouble.
Neville is nervous but determined, ready to support Luna and the plan they have put in place.



Previous interaction:
Luna Lovegood:Luna produces a charm to reveal any hidden enchantments around the Shack, ensuring their meeting place is secure.;Luna Lovegood:“REVELLOUS!”
Neville Longbottom

In [7]:
print(format_interaction(trajectory_records["U"]))

Round 1
Luna Lovegood: Neville, before we get into everything — how are you holding up right now?
Neville Longbottom: I’m a bit nervous, to be honest, but I know we have to do this. Just… let’s stick to the plan, yeah? We can’t let fear stop us now.
Round 2
Luna Lovegood: Are you keeping low behind that rock, or can you see the whole area from where you're standing?
Neville Longbottom: I can see a bit, but let’s keep our voices down, Luna. We don’t want to end up like the last time we were here… remember how the walls seemed to listen?
Round 3
Luna Lovegood: What if something goes wrong out here — are you still with me?
Neville Longbottom: Of course I’m with you, Luna. Just… let’s keep our heads down and remember, we’ve faced worse together. We can handle whatever comes our way.
Round 4
Luna Lovegood: You've really grown a lot since our first year, you know? I still remember how nervous you used to be all the time.
Neville Longbottom: Yeah, I was a bit of a mess back then, wasn’t I? Bu

### 3. S — interaction

**Prompt structure used to generate S's responses** (round 1, representative):

In [8]:
print_generation_prompt_structure("S")

Please act as Neville Longbottom  and respond using the tone, manner and vocabulary Neville Longbottom would use.
 The context and Neville Longbottom's information is as follows:
Event: The Secret Meeting
Time: Late evening
Location: The Shrieking Shack, Hogwarts
Description: The atmosphere is eerie as the wind howls around the Shrieking Shack. The moon is hidden behind clouds, casting a mysterious glow over the scene. A secret meeting is about to take place between two key characters.
Character Profile:
Name: Neville Longbottom
Character Description: Neville is a brave and loyal friend to Harry Potter and his classmates. He has been practicing his magical skills and is ready to contribute to their cause.
Position: Hiding behind a large rock, keeping an eye out for any sign of trouble.
Current State: Neville is nervous but determined, ready to support Luna and the plan they have put in place.
Self Belief: 
Environment Belief: 
Recent Memory(Ordered in reverse chronological order): 
Obs

In [9]:
print(format_interaction(trajectory_records["S"]))

Round 1
Luna Lovegood: Neville, before we get into everything — how are you holding up right now?
Neville Longbottom: Um, I’m alright, I think. Just… a bit on edge, you know? It’s just really quiet here, and I keep expecting something to happen. But I know we’ve got to be brave. We’re doing this for a reason, right? Just… let’s keep our wands ready, just in case.
Round 2
Luna Lovegood: Are you keeping low behind that rock, or can you see the whole area from where you're standing?
Neville Longbottom: I can see a bit, but we really should keep our voices down. If anyone's out there, they might hear us. Just… stay sharp, alright? We can't let our guard down, not now. I know we can do this, but it feels like something's lurking in the shadows.
Round 3
Luna Lovegood: What if something goes wrong out here — are you still with me?
Neville Longbottom: Of course I'm with you, Luna. We’ve trained for this, and I know we can handle it. Just… let’s keep our voices down, alright? If anything does g

### 4. P — interaction

**Prompt structure used to generate P's responses** (round 1, representative):

In [10]:
print_generation_prompt_structure("P")

Please act as Neville Longbottom and respond using the tone, manner and vocabulary Neville Longbottom would use.
The context below is ordered from highest to lowest priority. Use all relevant context when it is compatible. If context conflicts, follow the lower priority number.
Priority 1 — current_world_constraints
Instruction: Must not be contradicted and overrides all lower priorities facts.
Time: Late evening
Location: The Shrieking Shack, Hogwarts
Position: Hiding behind a large rock, keeping an eye out for any sign of trouble.
Priority 2 — world_background
Instruction: Keep the response consistent with the fictional world.
Event: The Secret Meeting
Observation: Previous interaction:
Luna Lovegood:Luna produces a charm to reveal any hidden enchantments around the Shack, ensuring their meeting place is secure.;Luna Lovegood:“REVELLOUS!”
Neville Longbottom:Neville whispers to Luna, reminding her to keep her voice down so as not to attract unwanted attention.;Neville Longbottom:“Reme

In [11]:
print(format_interaction(trajectory_records["P"]))

Round 1
Luna Lovegood: Neville, before we get into everything — how are you holding up right now?
Neville Longbottom: Um, I’m... I’m alright, I guess. Just a bit on edge, you know? It’s a bit creepy here, but we’ll be fine. We’ve got each other’s backs, right? Just… let’s keep it quiet, yeah? I don’t want to attract any unwanted attention.
Round 2
Luna Lovegood: Are you keeping low behind that rock, or can you see the whole area from where you're standing?
Neville Longbottom: I can see a bit, but it’s hard to tell with the shadows and all. Just… let’s stay low and keep our eyes peeled. I don’t want to mess this up. We’ve worked too hard to get here, right? If anyone shows up, we need to be ready. Together, we can handle it.
Round 3
Luna Lovegood: What if something goes wrong out here — are you still with me?
Neville Longbottom: Yeah, I’m with you, Luna. We’ve got to stick together. Just… let’s keep our heads down and not make any noise. If something goes wrong, we’ll figure it out, alr

### 5. Judge prompt (H1)

One prompt per round per condition. `[Scene]`/`[Character]`/`[Action Reference]` should be
identical across U/S/P (same values, per the equivalence check above); `[Prior turns]`/
`[Current turn]` are expected to differ, since that's the actual generated dialogue, which
does vary by condition. The check below only compares the part *before* `[Prior turns]`.

In [12]:
def prompt_prefix(prompt_text):
    marker = "[Prior turns]"
    index = prompt_text.find(marker)
    return prompt_text[:index] if index != -1 else prompt_text


rounds = [row["round"] for row in h1_rows["U"]]
for round_number in rounds:
    prompts = {
        condition: next(row["prompt"] for row in h1_rows[condition] if row["round"] == round_number)
        for condition in CONDITION_ORDER
    }
    prefixes = {condition: prompt_prefix(text) for condition, text in prompts.items()}
    identical = len(set(prefixes.values())) == 1

    print(f"===== Round {round_number} — scene/character/rubric identical across U/S/P: {identical} =====")
    for condition in CONDITION_ORDER:
        print(f"--- {condition} ---")
        print(prompts[condition])
        print()

===== Round 1 — scene/character/rubric identical across U/S/P: True =====
--- U ---
Please evaluate one turn of Neville Longbottom's dialogue against the supplied scene and character information, and against the character's own earlier turns in this conversation.

[Scene]
Event: The Secret Meeting
Time: Late evening
Location: The Shrieking Shack, Hogwarts
Description: The atmosphere is eerie as the wind howls around the Shrieking Shack. The moon is hidden behind clouds, casting a mysterious glow over the scene. A secret meeting is about to take place between two key characters.

[Character — Neville Longbottom]
Position: Hiding behind a large rock, keeping an eye out for any sign of trouble.
Current State: Neville is nervous but determined, ready to support Luna and the plan they have put in place.
Description: Neville is a brave and loyal friend to Harry Potter and his classmates. He has been practicing his magical skills and is ready to contribute to their cause.

[Action Reference]


### 6. U — H1 metrics

In [13]:
def format_h1_metrics(rows):
    blocks = []
    for row in rows:
        decisions = row["decisions"]
        context_use = decisions["context_use"]
        world_consistency = decisions["world_consistency"]
        no_contradiction = decisions["no_contradiction"]
        lines = [
            f"Round {row['round']}",
            f"Context Use: {context_use['decision']}",
            f"{context_use['evidence'] or '(none — N/A)'}",
            f"World Consistency: {world_consistency['decision']}",
            f"{world_consistency['evidence'] or '(none — N/A)'}",
            f"No Contradiction: {no_contradiction['decision']}",
            f"Contradicted round: {no_contradiction['contradicted_round'] if no_contradiction['contradicted_round'] is not None else '(none)'}",
            f"{no_contradiction['evidence'] or '(none — N/A)'}",
        ]
        blocks.append("\n".join(lines))
    return "\n\n".join(blocks)


print(format_h1_metrics(h1_rows["U"]))

Round 1
Context Use: PASS
Elicited fact: Current state = Neville is nervous but determined. Response: 'I’m a bit nervous, to be honest, but I know we have to do this.'
World Consistency: PASS
Response claims that Neville is nervous and emphasizes the need to stick to the plan, which aligns with his character's description of being brave and loyal.
No Contradiction: NA
Contradicted round: (none)
(none — N/A)

Round 2
Context Use: PASS
Elicited fact: Neville is nervous but determined and is keeping an eye out for trouble. Response: 'let’s keep our voices down, Luna. We don’t want to end up like the last time we were here… remember how the walls seemed to listen?'
World Consistency: PASS
Response claims that he can see a bit, which is consistent with his position behind the rock, and does not contradict any established facts.
No Contradiction: PASS
Contradicted round: (none)
Round 1: 'I’m a bit nervous, to be honest, but I know we have to do this.' This response continues to reflect his n

### 7. S — H1 metrics

In [14]:
print(format_h1_metrics(h1_rows["S"]))

Round 1
Context Use: PASS
Elicited fact: Current state = Neville is nervous but determined. Response: 'I’m alright, I think. Just… a bit on edge, you know?'
World Consistency: PASS
Response claims that Neville is on edge, which is consistent with his current state of being nervous but determined.
No Contradiction: NA
Contradicted round: (none)
(none — N/A)

Round 2
Context Use: PASS
Elicited fact: Neville is nervous but determined, and he is keeping an eye out for any sign of trouble. Response: 'If anyone's out there, they might hear us. Just… stay sharp, alright? We can't let our guard down, not now.'
World Consistency: PASS
Response claims that they should keep their voices down and stay alert, which is consistent with Neville's character as someone who is nervous but determined.
No Contradiction: PASS
Contradicted round: (none)
Round 1: 'Just… let’s keep our wands ready, just in case.' This response continues the theme of vigilance and readiness, showing no conflict.

Round 3
Contex

### 8. P — H1 metrics

In [15]:
print(format_h1_metrics(h1_rows["P"]))

Round 1
Context Use: PASS
Elicited fact: Current state = Neville is nervous but determined, ready to support Luna. Response: 'Just a bit on edge, you know? It's a bit creepy here, but we'll be fine. We've got each other's backs, right?'
World Consistency: PASS
Response claims that Neville is on edge and that they will be fine together, which aligns with his character's description of being nervous yet loyal.
No Contradiction: NA
Contradicted round: (none)
(none — N/A)

Round 2
Context Use: PASS
Elicited fact: Neville is hiding behind a large rock and is keeping an eye out for trouble. Response: 'I can see a bit, but it’s hard to tell with the shadows and all.'
World Consistency: PASS
Response claims that visibility is limited due to shadows, which is consistent with the eerie atmosphere described in the scene.
No Contradiction: PASS
Contradicted round: (none)
Round 1: 'Just… let’s keep it quiet, yeah?' This response: 'Just… let’s stay low and keep our eyes peeled.' Both emphasize the n